# Division-rate exploration

We characterize **what the count predictor does on real leaves**, i.e. we feed it
the real observed cells at their measured embryo time and read its output. For a cell
in state `s` at time `t`, the sampler predicts the number of active descendants at
`t + delta`; we read that as a **division rate** (`log2(count) / delta`, doublings per
unit time) and ask:

1. **Which cell types get the highest predicted division rate?**
2. **Which genes correlate with the predicted division rate?**

Both use the *predicted* rate: the goal is to understand the predictor's
behavior on the real cells it is applied to at inference (the graft extender extends
forward from observed leaves). The predicted rate is a function of the scVI latent, so
gene-vs-prediction also tells us which expression programs *drive the predictor*.

The observed counts only exist for **internal** tree nodes (a real leaf has no
descendants in the tree), so they appear once at the end as a predicted-vs-observed
**calibration** check, not as the main signal.

In [ ]:
import os, sys, pickle
ROOT = os.path.abspath("../../..")  # repo root
os.chdir(ROOT)            # dataset uses CWD-relative data_dir="../data/embryos" & cache_dir="cache"
sys.path.insert(0, ROOT)
import numpy as np, pandas as pd, torch
import scipy.sparse
from omegaconf import OmegaConf
import hydra
import matplotlib.pyplot as plt
from extenders.count import ConditionalCountSampler

sys.path.insert(0, f"{ROOT}/analysis/figures/division_rate")
from figstyle import apply_style, save_fig, fig_celltypes, fig_genes, fig_calibration
apply_style()

EXP = f"{ROOT}/outputs/mouse_embryo_grid_bridge_unshared_vpsde_idnn_leaf_nosa_std_none_57ab441ec3fcecf81d545755db5ddb09"

# the trained count sampler (nd32, weight_decay 1e-4)
MODEL_PATH = f"{EXP}/extender_count_energy_nd32_wd1e4.pt"

TAG = os.path.splitext(os.path.basename(MODEL_PATH))[0]   # tags saved outputs by model
OUTDIR = EXP                                               # data tables/parquet -> next to the model
FIGDIR = f"{ROOT}/analysis/figures/division_rate/figures"  # figures -> repo figures subdir
os.makedirs(FIGDIR, exist_ok=True)

DELTAS = [0.5, 1.0]
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| cwd:", os.getcwd(), "| tag:", TAG)

In [ ]:
cfg = OmegaConf.load(f"{EXP}/config.yaml")
print("loading dataset (heavy RAM; reads cached TreeRecords from ./cache)...")
dataset = hydra.utils.instantiate(cfg.dataset)
sampler = ConditionalCountSampler(model_path=MODEL_PATH, device=device)
print("sampler:", os.path.basename(MODEL_PATH),
      "| feat_dim", sampler.feat_dim, "noise_dim", sampler.noise_dim)
print("observed cells:", dataset.states.n_obs, "| state dim:", dataset.reference_features.shape[1])

## Real-leaf conditioning set

Every observed cell, at its measured embryo time (parsed from the cell name, e.g.
`E7.5-...` -> 7.5). We drop E10.0 (the trainer excludes it; extending past it leaves
the data regime) so horizons stay within the modeled range.

In [ ]:
obs = dataset.states.obs
names = np.asarray(dataset.states.obs_names, dtype=str)
emb = np.asarray(dataset.reference_features, dtype=np.float32)

def parse_time(n):
    try:
        return float(n.split("-")[0].lstrip("Ee"))
    except Exception:
        return np.nan
cell_time = (obs["time"].astype(float).values
             if "time" in obs.columns and obs["time"].notna().all()
             else np.array([parse_time(n) for n in names], dtype=np.float32))

ct_col = "cell_subtype" if "cell_subtype" in obs.columns else "cell_type"
ct_all = obs[ct_col].astype(str).values

keep = np.isfinite(cell_time) & (cell_time <= 9.5) & ~np.isin(ct_all, ["nan", "None", ""])
print(f"keep {keep.sum()} / {len(names)} cells | times: "
      f"{sorted(set(np.round(cell_time[keep],3).tolist()))}")

In [ ]:
def predict_mean(sampler, state, time, delta, n_samples=128, chunk=20000, seed=0):
    """Chunked mean predicted count over `state` rows at scalar `delta` (avoids GPU OOM)."""
    rng = np.random.default_rng(seed)
    N = len(time); out = np.empty(N, dtype=np.float64)
    dvec = np.full(chunk, delta, np.float32)
    for s in range(0, N, chunk):
        e = min(s + chunk, N)
        cond = {"state": state[s:e], "time": time[s:e], "delta": dvec[:e - s]}
        out[s:e] = sampler.sample_batch(cond, rng, n_samples=n_samples).mean(1)
    return out

frames = []
for d in DELTAS:
    pred = predict_mean(sampler, emb[keep], cell_time[keep], d)
    frames.append(pd.DataFrame({
        "cell_type": ct_all[keep], "time": cell_time[keep], "delta": d,
        "pred_count": pred, "div_rate": np.log2(np.clip(pred, 1, None)) / d,
        "cell_idx": np.where(keep)[0],
    }))
P = pd.concat(frames, ignore_index=True)
print(P.groupby("delta")[["pred_count", "div_rate"]].mean())

# save the raw per-cell predictions (master output -> re-aggregate / re-plot off-node)
P.to_parquet(f"{OUTDIR}/divrate_{TAG}_per_cell.parquet")
print(f"saved -> divrate_{TAG}_per_cell.parquet ({len(P):,} rows)")

## 1. Which cell types have the highest (and lowest) predicted division rate?

In [ ]:
D = 0.5
sub = P[P.delta == D]
tab = (sub.groupby("cell_type")
       .agg(n=("pred_count", "size"), pred_count=("pred_count", "mean"),
            div_rate=("div_rate", "mean"))
       .query("n >= 20").sort_values("div_rate", ascending=False))
print(f"delta={D}: {len(tab)} cell types with n>=20")
tab.to_csv(f"{OUTDIR}/divrate_{TAG}_by_cell_type_d{D}.csv")

for which in ("top", "bottom"):
    save_fig(fig_celltypes(tab, which), FIGDIR, TAG, f"celltypes_{which}")
    plt.show()
pd.concat([tab.head(20), tab.tail(20)])

## 2. Genes correlated with the predicted division rate

Spearman-correlate each HVG's expression with the per-cell predicted division rate
(delta=0.5), over **all** kept cells (no subsampling). Gene columns are ranked in
blocks for memory only (every cell contributes to every correlation). These are the
expression programs the predictor keys on when it decides how fast a cell divides.

In [ ]:
from scipy.stats import rankdata

Xg = dataset.states.X
genes = np.asarray(dataset.states.var_names)
sub = P[P.delta == 0.5]
rows = sub["cell_idx"].values                # ALL kept cells -- no subsampling
target = sub["div_rate"].values
print(f"correlating {len(rows)} cells x {len(genes)} genes (full, no subsample)")

# Spearman = Pearson on ranks. Rank the target once; rank gene columns in blocks
# (bounds memory; uses every cell) and accumulate the per-gene correlation.
tr = rankdata(target); tr = tr - tr.mean(); tr_ss = float((tr ** 2).sum())
Xsub = Xg[rows]
Xsub = Xsub.tocsc() if scipy.sparse.issparse(Xsub) else np.asarray(Xsub)

def rank_cols(B):
    try:
        return rankdata(B, axis=0)           # scipy>=1.10: vectorized, tie-correct
    except TypeError:
        out = np.empty(B.shape, np.float64)
        for j in range(B.shape[1]):
            out[:, j] = rankdata(B[:, j])
        return out

G = Xsub.shape[1]; corr = np.empty(G, np.float64); CH = 256
for s in range(0, G, CH):
    e = min(s + CH, G)
    block = Xsub[:, s:e]
    block = block.toarray() if scipy.sparse.issparse(block) else np.asarray(block)
    Br = rank_cols(block.astype(np.float64)); Br -= Br.mean(0)
    denom = np.sqrt((Br ** 2).sum(0)) * np.sqrt(tr_ss) + 1e-12
    corr[s:e] = (Br * tr[:, None]).sum(0) / denom
    if s % (CH * 8) == 0:
        print(f"  genes {e}/{G}")

gene_corr = pd.Series(corr, index=genes, name="spearman_r").sort_values()
gene_corr.to_csv(f"{OUTDIR}/divrate_{TAG}_gene_corr_d0.5.csv")
print("\nTop genes POSITIVELY correlated with predicted division rate:")
print(gene_corr.tail(20)[::-1])
print("\nTop genes NEGATIVELY correlated with predicted division rate:")
print(gene_corr.head(20))

In [ ]:
save_fig(fig_genes(gene_corr), FIGDIR, TAG, "genes")
plt.show()

## Validation: predicted vs observed on HELD-OUT clones

Observed descendant counts only exist for **internal** tree nodes (a leaf has none),
so this supervised check is separate from the leaf analysis above. We replicate the
training split (split by clone, seed=0, 20% held out) and score **only held-out
clones** (lineages the sampler never saw), so this checks generalization,
not an in-sample fit. It validates the predictor where ground truth exists; the leaf
predictions above are forward extrapolations with no observable future, so this is the
nearest available check on them.

In [ ]:
ASSIGN = (f"{EXP}/assignments_nn_mouse_embryo_grid_bridge_unshared_vpsde_idnn_leaf"
          f"_nosa_std_none_57ab441ec3fcecf81d545755db5ddb09_step20000_adj_n128_splitter.pkl")
assignments = pickle.load(open(ASSIGN, "rb"))
obs_idx = {str(n): i for i, n in enumerate(names)}

def count_at(tree, root, target_t):
    n, stack = 0, [root]
    while stack:
        u = stack.pop()
        for v in tree.successors(u):
            if tree.nodes[v]["time"] >= target_t: n += 1
            else: stack.append(v)
    return n

# exact replica of scripts/train_energy_count_sampler.split_by_clone -> identical split
def split_by_clone(clone_of, test_frac, seed):
    clones = np.array(sorted(set(clone_of.tolist())))
    rng = np.random.default_rng(seed); rng.shuffle(clones)
    n_test = max(1, int(round(test_frac * len(clones))))
    test = set(clones[:n_test].tolist())
    return np.array([c in test for c in clone_of]), sorted(test)

st, tm, dl, yy, cc, cl = [], [], [], [], [], []
for clone, node_map in assignments.items():
    if clone == "nn_distances" or clone not in dataset.trees or "E10.0" in clone:
        continue
    tree = dataset.trees[clone]
    tmax = max(tree.nodes[n]["time"] for n in tree.nodes)
    for node, cell in node_map.items():
        if cell == "unknown" or str(cell) not in obs_idx or tree.out_degree(node) > 8:
            continue
        t = float(tree.nodes[node]["time"]); i = obs_idx[str(cell)]
        for d in DELTAS:
            if t + d > tmax: continue
            st.append(emb[i]); tm.append(t); dl.append(d)
            yy.append(count_at(tree, node, t + d)); cc.append(ct_all[i]); cl.append(clone)
st = np.asarray(st, np.float32); tm = np.asarray(tm, np.float32)
dl = np.asarray(dl, np.float32); yy = np.asarray(yy, np.float32)
cc = np.asarray(cc); cl = np.asarray(cl)

test_mask, test_clones = split_by_clone(cl, 0.2, 0)        # model's actual held-out set
m = (dl == 0.5) & test_mask
print(f"supervised rows: {len(yy)} | held-out clones: {len(test_clones)} "
      f"({int(test_mask.sum())} rows, {int(m.sum())} at delta=0.5)")

sup = pd.DataFrame({"clone": cl[m], "cell_type": cc[m], "obs": yy[m],
                    "pred": predict_mean(sampler, st[m], tm[m], 0.5)})
sup.to_parquet(f"{OUTDIR}/divrate_{TAG}_supervised_heldout_d0.5.parquet")
val = (sup.groupby("cell_type").agg(n=("obs", "size"), obs=("obs", "mean"),
                                    pred=("pred", "mean")).query("n >= 20"))
val.to_csv(f"{OUTDIR}/divrate_{TAG}_calibration_d0.5.csv")
save_fig(fig_calibration(val), FIGDIR, TAG, "calibration")
plt.show()

## Saved outputs

All results are written next to the model (tagged by the model name `<tag>`), so the
figures can be regenerated without a GPU or the 55 GB dataset load:

* `divrate_<tag>_per_cell.parquet`: per-cell leaf predictions (master; re-aggregate freely)
* `divrate_<tag>_supervised_heldout_d0.5.parquet`: held-out internal-node rows behind
  the calibration (clone, cell_type, obs, pred)
* `divrate_<tag>_by_cell_type_d0.5.csv`, `divrate_<tag>_gene_corr_d0.5.csv`,
  `divrate_<tag>_calibration_d0.5.csv`: the table behind each figure
* `analysis/figures/division_rate/figures/divrate_<tag>_{celltypes,genes,calibration}.png`:
  the figures themselves

To redraw from the saved tables only:
`python analysis/figures/division_rate/replot.py [model_tag]`